In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier

df=pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')
df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1, inplace=True)
df['Age'] = df['Age'].fillna(df['Age'].mean())
df.dropna(subset=['Embarked'], inplace=True)
for atrib in ['Sex', 'Embarked']:
  df[atrib] = LabelEncoder().fit_transform(df[atrib].values)
target = 'Survived'
df[target]=pd.Categorical(df[target])
x = df.drop(target, axis=1)
y = df[target]
RANDOM_SEED=28

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=RANDOM_SEED)

In [ ]:
# AdaBoost

In [ ]:
clf = AdaBoostClassifier(random_state = RANDOM_SEED)
clf.fit(x_train, y_train)

In [ ]:
# Validación

y_pred = clf.predict(x_test)


In [ ]:
print("accuray score: ", accuracy_score(y_test, y_pred))

In [ ]:
pd.Series(clf.feature_importances_, index=clf.feature_names_in_).nlargest(clf.feature_names_in_.shape[0]).plot(kind='bar')

In [ ]:
# Ahora con árboles de clasificación (stumps) con max_depth=2

In [ ]:
clf = AdaBoostClassifier(random_state = RANDOM_SEED, estimator = DecisionTreeClassifier(max_depth=2))
clf.fit(x_train, y_train)

In [ ]:
y_pred = clf.predict(x_test)

print("accuray score: ", accuracy_score(y_test, y_pred))

In [ ]:
# GridSearchCV para intentar encontrar combinación óptima de hiperparámetros.

In [ ]:
param_dist = {
  'estimator':          [DecisionTreeClassifier(random_state=RANDOM_SEED, max_depth=1), DecisionTreeClassifier(random_state=RANDOM_SEED, max_depth=2)],
  #'estimator':            [DecisionTreeClassifier(random_state=RANDOM_SEED)],
  #'estimator__max_depth': [1, 2],
  'n_estimators':         [25, 50, 100, 200],  # Valor por defecto: 50.
  'learning_rate':        [0.25, 0.5, 1, 2, 4, 8],   # Valor por defecto: 1. "trade-off between the learning_rate and n_estimators parameters". Se entiende que con más clasificadores debería ser menor el valor de learning_rate.
}

In [ ]:
grid_clf = GridSearchCV(estimator = AdaBoostClassifier(), param_grid = param_dist, cv = 5)

grid_clf.fit(x, y)


In [ ]:
print(grid_clf.best_params_)
print(grid_clf.best_score_)

In [ ]:
# Ahora se entrena el mejor clasificador encontrado y se valida con validación simple

clf = grid_clf.best_estimator_

clf.fit(x_train, y_train)

In [ ]:
# Validación

y_pred = clf.predict(x_test)

print("accuray score: ", accuracy_score(y_test, y_pred))

In [ ]:
pd.Series(clf.feature_importances_, index=clf.feature_names_in_).nlargest(clf.feature_names_in_.shape[0]).plot(kind='bar')